# GA vs random frozen winner shared-test evaluation

Bu notebook yeni model egitmez. Full-validation ile onceden secilmis GA ve random checkpointlerini ayni 19.500 shared-test goruntusunde degerlendirir. Ilk uc kod hucresi test goruntusu acmadan repo, validation sonucu ve checkpoint hashlerini dondurur. Ciktilari Codex ile incelemeden kilitli test hucresini acma.


In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, time, zipfile

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab icin hazirlanmistir.'
from google.colab import drive, files
drive.mount('/content/drive')
print('Drive baglandi.')


## 1. Repo ve dondurulmus validation girdileri
Bu asamada dataset ZIP'i acilmaz ve test goruntusu okunmaz.


In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
CONFIRMATION_ROOT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_full_confirmation_v1')
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_shared_test_v1')
TOP_LOG = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_shared_test_v1_console.log')
REVIEW_ZIP = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_shared_test_v1_review_package.zip')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
EXTRACT_MARKER = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
FROZEN_RUNNER_SHA256 = 'd11e0f3cac7b0a056ce87a163f83e8aa043206408352623e83170dd854bb7419'
FROZEN_CONFIG_SHA256 = '9ac37b7c8d400eb363fecfc4801d4df835ab4f629e3a371a0a5e67998dec934e'
FROZEN_SPLIT_SHA256 = '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171'
FROZEN_TAXONOMY_SHA256 = '63820de352eb442cacf7f9701ac6006ff3c16a26642eae5042015941d8d8dd1b'
FROZEN_CONFIRMATION_SUMMARY_SHA256 = '37c5d8a2f353642403a0d5469122a01c2c82fb915fd60b5efb0f7bd7744d91e1'
EXPECTED_GROUPS = ('ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer', 'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
RUNNER_PATH = PROJECT_ROOT / 'src/evaluate_ga_random_shared_test.py'
CONFIG_PATH = PROJECT_ROOT / 'config/ga_random_shared_test_v1.json'
SPLIT_MANIFEST = PROJECT_ROOT / 'research/splits/practical_leakage_reduced_split_v1.csv'
TAXONOMY_PATH = PROJECT_ROOT / 'config/label_taxonomy.json'
for path, expected in ((RUNNER_PATH, FROZEN_RUNNER_SHA256), (CONFIG_PATH, FROZEN_CONFIG_SHA256), (SPLIT_MANIFEST, FROZEN_SPLIT_SHA256), (TAXONOMY_PATH, FROZEN_TAXONOMY_SHA256), (CONFIRMATION_ROOT / 'confirmation_summary.json', FROZEN_CONFIRMATION_SUMMARY_SHA256)):
    assert path.is_file(), f'Eksik frozen girdi: {path}'
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, f'Hash uyusmuyor: {path} expected={expected} actual={actual}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Git commit:', commit)
print('Confirmation root:', CONFIRMATION_ROOT)
print('Shared-test output:', OUTPUT_DIR)
print('Test goruntusu henuz acilmadi.')


## 2. Checkpoint freeze preflight
Iki validation-selected `best_model.pth` dosyasinin hash'i ve ic sozlesmesi kontrol edilir. Dataset ZIP'i veya test goruntusu acilmaz.


In [ ]:
import torch, torchvision
assert torch.cuda.is_available(), 'GPU secilmemis; T4 runtime ac.'
freeze_command = [
    sys.executable, '-u', str(RUNNER_PATH),
    '--mode', 'freeze',
    '--confirmation-root', str(CONFIRMATION_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--split-manifest', str(SPLIT_MANIFEST),
    '--taxonomy', str(TAXONOMY_PATH),
    '--config', str(CONFIG_PATH),
]
subprocess.run(freeze_command, cwd=PROJECT_ROOT, check=True)
checkpoint_freeze = json.loads((OUTPUT_DIR / 'checkpoint_freeze.json').read_text(encoding='utf-8'))
assert checkpoint_freeze['status'] == 'ga_random_shared_test_v1_checkpoints_frozen_test_not_opened'
assert checkpoint_freeze['test_opened_during_freeze'] is False
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GA checkpoint:', checkpoint_freeze['checkpoints']['ga'])
print('Random checkpoint:', checkpoint_freeze['checkpoints']['random'])
print('Preflight tamamlandi. Test goruntusu acilmadi; egitim yapilmadi.')


## 3. Kilitli shared-test degerlendirmesi
Ilk uc kod hucresinin ciktilarini Codex ile incelemeden `AUTHORIZE_SHARED_TEST` degerini degistirme. Bu hucre ZIP'i gecici Colab diskine acar ve iki frozen checkpointi ayni 19.500 test goruntusunde degerlendirir. Egitim, optimizer update veya test-temelli yeniden secim yapmaz. Kesinti olursa output klasorunu silmeden ayni hucreyi yeniden calistir; tamamlanmis aday hash ile atlanir.


In [ ]:
AUTHORIZE_SHARED_TEST = False
assert AUTHORIZE_SHARED_TEST, 'Shared test kilitli; checkpoint preflight Codex ile incelenmeden calismaz.'
summary_path = OUTPUT_DIR / 'shared_test_summary.json'
assert not summary_path.is_file(), 'Shared-test degerlendirmesi zaten tamamlanmis; yeniden calistirma.'
assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadi: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert free_gib >= 12.0, 'Dataset acmak icin en az 12 GiB yerel bos alan gerekli.'
if not EXTRACT_MARKER.is_file():
    LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print('ZIP Colab diskine cikariliyor; birkac dakika cikti gorunmeyebilir...')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(LOCAL_DATA_BASE)], check=True)
    EXTRACT_MARKER.write_text('complete\n', encoding='utf-8')
    print(f'ZIP cikarma tamamlandi: {(time.monotonic() - started) / 60:.1f} dakika')
dataset_candidates = (LOCAL_DATA_BASE / 'Multi Cancer' / 'Multi Cancer', LOCAL_DATA_BASE / 'Multi Cancer', LOCAL_DATA_BASE)
DATASET_ROOT = next((p for p in dataset_candidates if all((p / g).is_dir() for g in EXPECTED_GROUPS)), None)
assert DATASET_ROOT is not None, 'Sekiz ana klasoru iceren dataset koku bulunamadi.'
command = [
    sys.executable, '-u', str(RUNNER_PATH),
    '--mode', 'evaluate',
    '--confirmation-root', str(CONFIRMATION_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--data-dir', str(DATASET_ROOT),
    '--split-manifest', str(SPLIT_MANIFEST),
    '--taxonomy', str(TAXONOMY_PATH),
    '--config', str(CONFIG_PATH),
    '--workers', '4', '--device', 'cuda',
    '--authorize', 'FROZEN_GA_RANDOM_SHARED_TEST_V1',
]
print('Shared-test komutu:', ' '.join(command))
with TOP_LOG.open('a' if TOP_LOG.exists() else 'w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line); log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Shared test tamamlanamadi. Output klasorunu silme; logu Codex ile paylas.')


## 4. Kanit paketini dogrula ve indir
Bu hucre iki aday tamamlandiktan sonra calisir. Checkpointleri kopyalamaz; metrik, tahmin, rapor, confusion matrix ve hash kanitlarini paketler.


In [ ]:
summary = json.loads((OUTPUT_DIR / 'shared_test_summary.json').read_text(encoding='utf-8'))
assert summary['status'] == 'ga_random_shared_test_v1_complete'
assert summary['test_evaluated'] is True and summary['test_rows'] == 19500
assert summary['test_based_reselection_performed'] is False
hash_manifest = json.loads((OUTPUT_DIR / 'hash_manifest.json').read_text(encoding='utf-8'))
for relative, metadata in hash_manifest.items():
    path = OUTPUT_DIR / relative
    assert path.is_file(), f'Eksik artifact: {path}'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == metadata['sha256'], f'Hash uyusmuyor: {path}'
review_paths = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file() and path.name != 'review_artifact_hashes.json')
if TOP_LOG.is_file(): review_paths.append(TOP_LOG)
review_hashes = {}
for path in review_paths:
    arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
    review_hashes[arcname] = {'sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'size_bytes': path.stat().st_size}
review_manifest = OUTPUT_DIR / 'review_artifact_hashes.json'
review_manifest.write_text(json.dumps(review_hashes, indent=2), encoding='utf-8')
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_paths:
        arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
        archive.write(path, arcname=arcname)
    archive.write(review_manifest, arcname='review_artifact_hashes.json')
print('Durum:', summary['status'])
print('GA leaf metrics:', summary['candidate_results']['ga']['leaf_metrics'])
print('Random leaf metrics:', summary['candidate_results']['random']['leaf_metrics'])
print('GA eksi random:', summary['ga_minus_random'])
print('Paired comparison:', summary['paired_comparison'])
print('Review package:', REVIEW_ZIP)
print('Review package SHA-256:', hashlib.sha256(REVIEW_ZIP.read_bytes()).hexdigest())
files.download(str(REVIEW_ZIP))
